# Endpoint quickstart

Same recipe as `examples/inference/vllm-endpoint/serve.py`: vLLM serving `Qwen/Qwen3-0.6B` on one H100, behind token auth. The notebook checks the account, creates the endpoint, waits until the model answers, sends one chat request, reloads it by id, stops and starts it, and deletes it.

**Before you start**

- Run `uv sync` in the repo root, then start Jupyter from the same environment, for example `uv run --with jupyter jupyter lab`.
- You need a Nebius CLI profile (`nebius profile create`). The active profile supplies the project. The region comes from the profile or the project.
- You need quota for one on-demand H100 in `eu-north1`. For another region, change `gpu=`. Machine selection is in `docs/guides/machines.md`.

**An endpoint bills for compute while it runs, from create until you stop or delete it.** Interrupting a cell, a local `WaitTimeout`, and restarting the kernel all leave it running. If anything goes wrong after the create cell, run the last cell, **Delete**.

Jupyter already runs an event loop, so every call that talks to Nebius uses the awaited twin (`create_async`, `wait_async`, …). A synchronous `create()` in this kernel raises `ValidationError` and names the twin to use.

In [1]:
import json
import urllib.request

import nebius_serverless as ns

MODEL = "Qwen/Qwen3-0.6B"

## Check the account

`whoami_async` makes one read-only call and resolves the project and region exactly as `create` will. Nothing is created. `quiet=True` skips the summary line, which names the account.

In [2]:
who = await ns.whoami_async(quiet=True)

who["account"]["kind"], who["auth"]["source"], who["region"]

('user', 'profile', 'eu-north1')

## Recipe

An `EndpointSpec` is a local recipe. Constructing it does not contact Nebius.

`command` overrides the image entrypoint. Omit it to keep the entrypoint. Endpoints use token auth by default: a random bearer token is generated on create and is available as `endpoint.token`.

In [3]:
spec = ns.EndpointSpec(
    [
        "bash",
        "-c",
        "python3 -m vllm.entrypoints.openai.api_server "
        f"--model {MODEL} --host 0.0.0.0 --port 8000",
    ],
    image="vllm/vllm-openai:v0.19.1",
    gpu="h100",
    port=8000,
    disk="500Gi",
    name="qwen-tiny",
)

{
    "name": spec.name,
    "command": list(spec.command),
    "image": spec.image,
    "gpu": spec.gpu,
    "port": spec.port,
    "auth": spec.auth,
}

{'name': 'qwen-tiny',
 'command': ['bash',
  '-c',
  'python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-0.6B --host 0.0.0.0 --port 8000'],
 'image': 'vllm/vllm-openai:v0.19.1',
 'gpu': 'h100',
 'port': 8000,
 'auth': 'token'}

## Dry run

This resolves the project and region and returns the create request. It does not create an endpoint. `gpu` is sent as a selector, and Nebius picks the platform and preset at create. The token is shown as `"generated"`, never as a value.

In [4]:
await spec.create_async(dry_run=True)

{'image': 'vllm/vllm-openai:v0.19.1',
 'command': ['bash',
  '-c',
  'python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-0.6B --host 0.0.0.0 --port 8000'],
 'args': "-c 'python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-0.6B --host 0.0.0.0 --port 8000'",
 'ports': ['8000/http'],
 'project': 'project-e00…',
 'region': 'eu-north1',
 'gpu': 'h100',
 'pricing': 'on-demand',
 'auth': 'token',
 'token': 'generated',
 'env': [],
 'name': 'qwen-tiny',
 'disk': '500Gi',
 'shm': '16Gi'}

## Create

`create_async` returns an `Endpoint` as soon as Nebius accepts the endpoint and its id is known. It does not wait for the URL. **This cell starts a billable endpoint.**

Save `endpoint.id` so you can find the endpoint again. `endpoint.token` holds the generated token; after a kernel restart, `reveal_token_async()` on a reloaded handle fetches it again.

In [5]:
endpoint = await spec.create_async()

endpoint_id = endpoint.id
endpoint_id, endpoint.status

Submitted endpoint aiendpoint-e00… qwen-tiny


Token generated for aiendpoint-e00…. It is in endpoint.token; reveal_token() fetches it again. Use token_secret= to manage it in MysteryBox.


('aiendpoint-e00…', None)

## Wait until the model answers

This polls about every 5 seconds until the state is `RUNNING` with a managed `https://` URL, then until `/v1/models` answers with the bearer token. `RUNNING` alone means the tunnel is up, not that vLLM has loaded the model. Pulling the image and loading the model take a few minutes.

`timeout=` limits only the local wait. When it expires, the endpoint keeps running and billing. Re-run the cell to keep waiting.

In [6]:
try:
    await endpoint.wait_async(timeout=30 * 60, probe="/v1/models")
except ns.WaitTimeout as exc:
    print(f"Stopped waiting: {exc}")
    print("Re-run this cell to keep waiting, or delete it in the last cell.")
else:
    endpoint.display()

aiendpoint-e00… PROVISIONING · 0.2s


aiendpoint-e00… STARTING · 102.8s


aiendpoint-e00… IMAGE_PULLING · 148.9s


aiendpoint-e00… STARTING · 282.3s


✓ aiendpoint-e00… RUNNING · 287.4s


  https://port8000-….tunnel.applications.eu-north1.nebius.cloud


... (9 outputs left out)

  shm       16 GiB


  subnet    vpcsubnet-e00…


  created   2026-11-19 22:04:40Z


  instance  computeinstance-e00…  10.0.0.4  RUNNING


## Call the model

This client does not wrap inference. Send HTTP yourself to `endpoint.url` with the bearer token. vLLM serves the OpenAI-compatible API.

In [7]:
def chat(prompt):
    body = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 256,
    }
    request = urllib.request.Request(
        endpoint.url.rstrip("/") + "/v1/chat/completions",
        data=json.dumps(body).encode(),
        headers={
            "Authorization": f"Bearer {endpoint.token}",
            "Content-Type": "application/json",
        },
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        return json.load(response)["choices"][0]["message"]["content"]


print(chat("Say hello in five words."))

<think>
Okay, the user wants me to say "hello in five words." Let me think. First, "hello" is the main word. Then, "in five words" adds context. But how to phrase that concisely. Maybe "Hello, five words." That works. Let me check if there's a better way. Maybe "Hello, five words." Or "Hello, five words." Either way, it's five words. I think that's correct.
</think>

Hello, five words.


## Reload by id

`endpoints.get_async` starts from an id string: after a kernel restart, in another notebook, or for an endpoint created from the CLI. It fetches once and never creates an endpoint. The reloaded handle has the URL; `reveal_token_async()` fetches the token from Nebius. The cell compares tokens instead of printing one.

In [8]:
loaded = await ns.endpoints.get_async(endpoint_id)
token_before = loaded.token
revealed = await loaded.reveal_token_async()

(
    loaded.status,
    loaded.url == endpoint.url,
    token_before,
    revealed == endpoint.token,
)

aiendpoint-e00… RUNNING


('RUNNING', True, None, True)

## Stop

`stop_async` asks Nebius to stop the endpoint and returns once the request is accepted. The endpoint keeps its id and configuration; `start_async` runs it again and only `delete_async` removes it. While it is stopped, compute and the disk are not billed.

The state passes through `STOPPING` to `STOPPED`. `wait_async(until="stopped")` blocks until then. A plain `wait_async()` could return on the `RUNNING` that is still reported right after the stop.

In [9]:
await endpoint.stop_async()
await endpoint.wait_async(until="stopped")

endpoint_id, endpoint.status

Stop accepted for aiendpoint-e00…


aiendpoint-e00… STOPPING · 0.1s


aiendpoint-e00… STOPPED · 46.4s


('aiendpoint-e00…', 'STOPPED')

## Start again

`start_async` runs the stopped endpoint again. It comes back at the same URL; the boot disk is not kept, so the model loads again. `wait_async` skips the `STOPPED` left over from before the start, and the probe waits for the model.

In [10]:
url_before = endpoint.url
await endpoint.start_async()
await endpoint.wait_async(timeout=30 * 60, probe="/v1/models")

endpoint.status, endpoint.url == url_before, chat("One word: ready?")

Start accepted for aiendpoint-e00…


aiendpoint-e00… STARTING · 0.1s


aiendpoint-e00… IMAGE_PULLING · 128.2s


✓ aiendpoint-e00… RUNNING · 266.8s


  https://port8000-….tunnel.applications.eu-north1.nebius.cloud


aiendpoint-e00… app not answering yet at /v1/models: 503


('RUNNING',
 True,
 '<think>\nOkay, the user is asking for a word that means "ready." They provided the word "ready," but maybe they want to confirm if that\'s the correct answer. Let me think.\n\nFirst, I need to check if "ready" is indeed the correct word. In English, "ready" is commonly used to indicate that something is prepared. For example, "I\'m ready to go." So, "ready" is a valid word. \n\nBut maybe the user is testing if I can identify the correct word. They might be looking for confirmation. Since I can\'t look up the word, I should make sure that "ready" is indeed the right answer. \n\nI should explain that "ready" is a common term and provide an example sentence to help clarify. Also, mention that if there\'s any confusion, they can ask again. That way, the user knows they can get more help if needed.\n</think>\n\nThe word "ready" is indeed a correct and commonly used term. It means prepared or set for an action. For example, "I\'m ready to go." If you have any further que

## Delete

`delete_async` removes the endpoint. It works on a running or a stopped endpoint and returns once the request is accepted. A later `get_async` shows `DELETING` or reports that the id is missing.

After a kernel restart, `endpoint` is gone. Run `await ns.endpoints.get_async("<endpoint id>")` and then `delete_async()` on the result.

In [11]:
await endpoint.delete_async()

try:
    await endpoint.get_async()
except ns.Error as exc:
    print(f"{endpoint_id} is gone: {exc}")
else:
    print(endpoint.status)

Delete accepted for aiendpoint-e00…


aiendpoint-e00… RUNNING


RUNNING
